In [1]:
# Cella 1: Setup
import sys
from pathlib import Path
import torch
from transformers import TrainingArguments, Trainer

ROOT = Path.cwd().resolve().parent
if str(ROOT / "src") not in sys.path:
    sys.path.append(str(ROOT / "src"))

from project_paths import get_paths
from teacher_finetune_headtail import (
    load_flat_dataset, TeacherModelConfig, build_teacher_model,
    build_teacher_tokenizer, build_collator, compute_metrics,
    get_llrd_optimizer_parameters, bf16_supported
)

paths = get_paths(ROOT)

c:\Users\cola0\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_ds = load_flat_dataset(paths.data_processed / "train_120k_ht.parquet")
val_ds = load_flat_dataset(paths.data_processed / "val_120k_ht.parquet")

MODEL_NAME = "bert-large-uncased"
tokenizer = build_teacher_tokenizer(MODEL_NAME)
collator = build_collator(tokenizer)

model_cfg = TeacherModelConfig(
    model_name = MODEL_NAME,
    gradient_checkpointing=True,
    hidden_dropout_prob = 0.1
)

model = build_teacher_model(model_cfg)

if model_cfg.gradient_checkpointing:
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()

KeyboardInterrupt: 

In [ ]:
LR_MAX = 2e-5
DECAY_RATE = 0.95
WEIGHT_DECAY = 0.01

optimizer_grouped_parameters = get_llrd_optimizer_parameters(
    model,
    learning_rate = LR_MAX,
    weight_decay = WEIGHT_DECAY,
    layer_decay = DECAY_RATE
)

optimizer = torch.optim.AdamW(optimizer_grouped_parameters)

args = TrainingArguments(
    output_dir=str(paths.checkpoints / "teacher_bert_large_llrd"),
    per_device_train_batch_size=4,   
    gradient_accumulation_steps=8,   
    num_train_epochs=2,              
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=250,
    save_strategy="steps",
    save_steps=250,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    bf16=bf16_supported(),
    fp16=not bf16_supported(),
    report_to="none"
)

class CustomTrainer(Trainer):
    def __init__(self, *args, pos_weight_value=None, **kwargs):
        super().__init__(*args, **kwargs)
        # Salviamo il valore del peso (es. 2.8)
        self.pos_weight_value = pos_weight_value

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # 1. Rimuoviamo labels dall'input per evitare calcoli automatici errati
        labels = inputs.pop("labels")
        
        # 2. Forward pass
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        # 3. Fix Dimensioni [Batch, 1] -> [Batch]
        if logits.shape != labels.shape:
            logits = logits.view(-1)
            labels = labels.view(-1)
            
        # 4. Configurazione Loss con Peso
        if self.pos_weight_value is not None:
            # Creiamo il tensore peso sullo stesso device (GPU) dei logits
            weight_tensor = torch.tensor([self.pos_weight_value], device=logits.device)
            loss_fct = torch.nn.BCEWithLogitsLoss(pos_weight=weight_tensor)
        else:
            loss_fct = torch.nn.BCEWithLogitsLoss()
            
        loss = loss_fct(logits, labels.float())
        
        return (loss, outputs) if return_outputs else loss


trainer = CustomTrainer(  
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, None),
    pos_weight_value=2.80
)

print("🚀 Starting LLRD Training (con Shape Fix)...")
trainer.train()

final_path = paths.checkpoints / "teacher_bert_large_final_headtail"
trainer.save_model(str(final_path))
print(f"Model saved to {final_path}")

🚀 Starting LLRD Training (con Shape Fix)...


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Auc
250,0.517200,0.492800,0.761333,0.258031,0.707386,0.157795,0.765192
500,0.505200,0.468992,0.786750,0.470734,0.677784,0.360583,0.794201
750,0.493000,0.475108,0.772667,0.280211,0.837539,0.168251,0.804768
1000,0.466900,0.457927,0.785917,0.390221,0.777673,0.260456,0.806307
1250,0.452500,0.448444,0.794583,0.504124,0.690358,0.397022,0.807378
1500,0.457100,0.453427,0.786417,0.387867,0.787585,0.257288,0.808518
1750,0.468500,0.448524,0.796417,0.493048,0.714372,0.376426,0.804438
2000,0.471800,0.444311,0.799833,0.549681,0.673095,0.464512,0.812018
2250,0.458900,0.449386,0.791167,0.577830,0.616906,0.543409,0.813453
2500,0.457000,0.464462,0.796167,0.440531,0.791941,0.305133,0.811782


Model saved to C:\Users\cola0\Desktop\nlp.project-Colangelo-2526\checkpoints\teacher_bert_large_final_headtail
